In [1]:
!git clone -b qmix https://github.com/yjkim717/smart-warehouse.git
!pip install rware gymnasium imageio pyyaml -q
import sys
sys.path.insert(0, '/content/smart-warehouse')

Cloning into 'smart-warehouse'...
remote: Enumerating objects: 392, done.
remote: Counting objects: 100% (392/392), done.
remote: Compressing objects: 100% (273/273), done.
remote: Total 392 (delta 151), reused 343 (delta 102), pack-reused 0 (from 0)
Receiving objects: 100% (392/392), 12.02 MiB | 12.39 MiB/s, done.
Resolving deltas: 100% (151/151), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.7 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import copy
import time
import datetime
import json
import csv
import yaml
from src.env.warehouse_env import WarehouseEnv


# ==========================================
# Directory Configuration (Colab-compatible)
# ==========================================
# Detect if running on Colab
try:
    import google.colab
    IS_COLAB = True
    BASE_DIR = '/content/smart-warehouse'
except ImportError:
    IS_COLAB = False
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))


# Automatically create a unique folder like "results_20260414_0645"
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

RESULTS_DIR = os.path.join(BASE_DIR, f'results_{timestamp}')
CHECKPOINTS_DIR = os.path.join(RESULTS_DIR, 'checkpoints')
LOGS_DIR = os.path.join(RESULTS_DIR, 'logs')
PLOTS_DIR = os.path.join(RESULTS_DIR, 'plots')
CONFIG_PATH = os.path.join(BASE_DIR, 'configs', 'env_config_4agents.yaml')


# ==========================================
# 1. GPU & XLA Configuration for Colab
# ==========================================
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Enable memory growth to avoid locking all VRAM instantly
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# Optional: Enable mixed precision if running on T4/A100 for further speedup
tf.keras.mixed_precision.set_global_policy('mixed_float16')


# ==========================================
# Environment & Configuration Setup
# ==========================================
# 1. Load the warehouse configuration
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Config file not found at: {CONFIG_PATH}")

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

# 2. Create necessary directories
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# 3. Instantiate the environment
env = WarehouseEnv(config)

# 3. Pull the dynamic dimensions
N_AGENTS = int(env.n_agents)
N_ACTIONS = int(env.action_dim)
N_OBS_DIM = int(env.obs_dim)

# Recompute N_FEATURES
N_FEATURES = N_AGENTS + N_OBS_DIM + N_ACTIONS

EYE_AGENTS = np.eye(N_AGENTS, dtype=np.float32)
EYE_ACTIONS = np.eye(N_ACTIONS, dtype=np.float32)

# Set up Hyperparameters
HIDDEN_DIM = 256
MIXER_HIDDEN_DIM = 256
TOTAL_TIMESTEPS = 10_000_000  # Increase to 10M to allow for more learning with the 4x update frequency
MAX_STEPS = config['env'].get('max_steps', 800)
GAMMA = 0.99
BATCH_SIZE = 256
MIN_REPLAY_SIZE = 1000
TARGET_UPDATE_INTERVAL = 200

# Calculate episodes based on total timesteps (assuming ~500 steps per episode)
NUM_EPISODES = TOTAL_TIMESTEPS // MAX_STEPS  # ~6,000 episodes

# ==========================================
# Evaluation Configuration (MAPPO-style)
# ==========================================
EVAL_INTERVAL = 1000
EVAL_EPISODES = 10

# ==========================================
# Epsilon Scheduling (adjusted for MAPPO-style annealing)
# ==========================================
epsilon_start = 1.0   # MUST start at 1.0 for Q-learning
epsilon_end = 0.05    # Lower floor to exploit better later
epsilon_decay_steps = int(NUM_EPISODES * 0.8)  # Decay over first 60% of training (Explores for 3,600 eps, exploits for 2,400 eps)


# ==========================================
# Network Definitions
# ==========================================
class DRQNAgent(tf.keras.Model):
    def __init__(self, hidden_dim, n_actions):
        super(DRQNAgent, self).__init__()
        self.hidden_dim = hidden_dim
        self.fc1 = tf.keras.layers.Dense(hidden_dim, activation='relu')
        self.gru = tf.keras.layers.GRU(hidden_dim, return_state=True)
        self.fc2 = tf.keras.layers.Dense(n_actions)

    @tf.function(reduce_retracing=True)
    def call(self, inputs, hidden_state):
        # FC layer expects (batch, features)
        x = self.fc1(inputs)

        # GRU expects (batch, timesteps, features). We process 1 timestep at a time.
        x = tf.expand_dims(x, axis=1)

        x, new_hidden_state = self.gru(x, initial_state=hidden_state)
        q_values = self.fc2(x)
        return q_values, new_hidden_state


class QMixer(tf.keras.Model):
    def __init__(self, n_agents, state_dim, embed_dim, hypernet_embed_dim):
        super(QMixer, self).__init__()
        self.n_agents = n_agents
        self.state_dim = state_dim
        self.embed_dim = embed_dim

        self.hyper_w_1 = tf.keras.Sequential([
            tf.keras.layers.Dense(hypernet_embed_dim, activation='relu'),
            tf.keras.layers.Dense(embed_dim * n_agents)
        ])
        self.hyper_b_1 = tf.keras.layers.Dense(embed_dim)

        self.hyper_w_2 = tf.keras.Sequential([
            tf.keras.layers.Dense(hypernet_embed_dim, activation='relu'),
            tf.keras.layers.Dense(embed_dim)
        ])
        self.hyper_b_2 = tf.keras.Sequential([
            tf.keras.layers.Dense(hypernet_embed_dim, activation='relu'),
            tf.keras.layers.Dense(1)
        ])

    @tf.function(reduce_retracing=True)
    def call(self, agent_qs, state):
        batch_size = tf.shape(agent_qs)[0]
        q_values = tf.reshape(agent_qs, (batch_size, 1, self.n_agents))

        w1 = tf.abs(self.hyper_w_1(state))
        w1 = tf.reshape(w1, (batch_size, self.n_agents, self.embed_dim))
        b1 = tf.reshape(self.hyper_b_1(state), (batch_size, 1, self.embed_dim))

        hidden = tf.nn.elu(tf.matmul(q_values, w1) + b1)

        w2 = self.hyper_w_2(state)
        w2 = tf.reshape(tf.abs(w2), (-1, self.embed_dim, 1))
        b2 = tf.reshape(self.hyper_b_2(state), (-1, 1, 1))

        q_tot = tf.matmul(hidden, w2) + b2
        return tf.squeeze(q_tot, axis=[1, 2])  # Remove dimensions 1 and 2 to get (batch_size,)


# ==========================================
# 2. XLA-Compiled Vectorized Action Selection
# ==========================================
# ---> FIX: Change jit_compile to reduce_retracing <---
@tf.function(reduce_retracing=True)
def get_actions_tf(q_values, epsilon):
    greedy_actions = tf.argmax(q_values, axis=1, output_type=tf.int32)
    random_actions = tf.random.uniform(shape=[tf.shape(q_values)[0]],
                                       minval=0,
                                       maxval=N_ACTIONS,
                                       dtype=tf.int32)
    rand_mask = tf.random.uniform(shape=[tf.shape(q_values)[0]], dtype=tf.float32) < epsilon
    return tf.where(rand_mask, random_actions, greedy_actions)

def get_global_state(env_observations):
    return np.concatenate(env_observations)


# ==========================================
# Replay Buffer Setup (Added Hidden States)
# ==========================================
D_MAX = 200000
D_obs = np.zeros((D_MAX, N_AGENTS, N_OBS_DIM), dtype=np.float32)
D_actions = np.zeros((D_MAX, N_AGENTS), dtype=np.int32)
D_rewards = np.zeros((D_MAX, N_AGENTS), dtype=np.float32)
D_next_obs = np.zeros((D_MAX, N_AGENTS, N_OBS_DIM), dtype=np.float32)
D_states = np.zeros((D_MAX, N_AGENTS * N_OBS_DIM), dtype=np.float32)
D_next_states = np.zeros((D_MAX, N_AGENTS * N_OBS_DIM), dtype=np.float32)
D_dones = np.zeros((D_MAX,), dtype=np.float32)
D_prev_act = np.zeros((D_MAX, N_AGENTS), dtype=np.int32)
D_hidden = np.zeros((D_MAX, N_AGENTS, HIDDEN_DIM), dtype=np.float32)      # NEW
D_next_hidden = np.zeros((D_MAX, N_AGENTS, HIDDEN_DIM), dtype=np.float32) # NEW
D_size = 0
D_ptr = 0

# ==========================================
# Model Instantiation (Updated for DRQN)
# ==========================================
agent_network = DRQNAgent(HIDDEN_DIM, N_ACTIONS)
mixer_network = QMixer(N_AGENTS, state_dim=(N_OBS_DIM * N_AGENTS), embed_dim=128, hypernet_embed_dim=64)

dummy_agent_input = tf.zeros((1, N_FEATURES))
dummy_hidden_state = tf.zeros((1, HIDDEN_DIM))
_, _ = agent_network(dummy_agent_input, dummy_hidden_state)

dummy_mixer_q = tf.zeros((1, N_AGENTS))
dummy_mixer_state = tf.zeros((1, N_OBS_DIM * N_AGENTS))
_ = mixer_network(dummy_mixer_q, dummy_mixer_state)

target_agent = DRQNAgent(HIDDEN_DIM, N_ACTIONS)
_, _ = target_agent(dummy_agent_input, dummy_hidden_state)
target_agent.set_weights(agent_network.get_weights())

target_mixer = QMixer(N_AGENTS, state_dim=(N_OBS_DIM * N_AGENTS), embed_dim=128, hypernet_embed_dim=64)
_ = target_mixer(dummy_mixer_q, dummy_mixer_state)
target_mixer.set_weights(mixer_network.get_weights())


# ==========================================
# 3. GPU-Accelerated Train Step (cuDNN Optimized)
# ==========================================
optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.0001)

# ---> FIX: Standard graph compilation instead of XLA <---
@tf.function(reduce_retracing=True)
def train_step(b_obs, b_prev_act_oh, b_actions, b_rewards, b_next_obs, b_states, b_next_states, b_dones, b_agent_id_oh, b_hidden, b_next_hidden):
    with tf.GradientTape() as tape:
        b_inputs = tf.concat([b_agent_id_oh, b_obs, b_prev_act_oh], axis=-1)
        flat_obs = tf.reshape(b_inputs, (-1, N_FEATURES))
        flat_hidden = tf.reshape(b_hidden, (-1, HIDDEN_DIM))

        b_act_oh = tf.one_hot(tf.squeeze(b_actions, axis=-1), N_ACTIONS, dtype=tf.float32)
        b_next_inputs = tf.concat([b_agent_id_oh, b_next_obs, b_act_oh], axis=-1)
        flat_next_obs = tf.reshape(b_next_inputs, (-1, N_FEATURES))
        flat_next_hidden = tf.reshape(b_next_hidden, (-1, HIDDEN_DIM))

        # Forward passes with hidden states
        all_q_vals, _ = agent_network(flat_obs, flat_hidden)
        all_q_vals = tf.reshape(all_q_vals, (BATCH_SIZE, N_AGENTS, N_ACTIONS))

        choose_action_q_vals = tf.squeeze(tf.gather(all_q_vals, b_actions, batch_dims=2), -1)
        q_tot = mixer_network(choose_action_q_vals, b_states)

        next_q_vals_online, _ = agent_network(flat_next_obs, flat_next_hidden)
        next_q_vals_online = tf.reshape(next_q_vals_online, (BATCH_SIZE, N_AGENTS, N_ACTIONS))
        next_actions_online = tf.expand_dims(tf.argmax(next_q_vals_online, axis=2, output_type=tf.int32), -1)

        target_q_vals, _ = target_agent(flat_next_obs, flat_next_hidden)
        target_q_vals = tf.reshape(target_q_vals, (BATCH_SIZE, N_AGENTS, N_ACTIONS))
        target_max_q_vals = tf.squeeze(tf.gather(target_q_vals, next_actions_online, batch_dims=2), -1)

        target_q_tot = target_mixer(target_max_q_vals, b_next_states)
        # ---> FIX: Cast network outputs to float32 BEFORE the math equation <---
        target_q_tot = tf.cast(target_q_tot, tf.float32)
        q_tot = tf.cast(q_tot, tf.float32)

        expected_q_tot = tf.stop_gradient(b_rewards + GAMMA * (1.0 - b_dones) * target_q_tot)

        # Loss and Optimization
        loss = tf.keras.losses.MSE(expected_q_tot, q_tot)
        loss = tf.reduce_mean(loss)

    vars_to_train = agent_network.trainable_variables + mixer_network.trainable_variables
    grads = tape.gradient(loss, vars_to_train)
    grads, _ = tf.clip_by_global_norm(grads, 0.5)
    optimizer.apply_gradients(zip(grads, vars_to_train))


# ==========================================
# 4. Streamlined Training Loop (MAPPO-style)
# ==========================================
training_rewards = []
training_deliveries = []
epsilon = epsilon_start

start_time = time.time()

# Best model tracking (like MAPPO)
best_eval_reward = float('-inf')

for episode in range(NUM_EPISODES):
    observations = env.reset()
    global_state = get_global_state(observations)
    previous_actions = np.zeros(N_AGENTS)

    # ---> NEW: Initialize hidden state at the start of the episode <---
    hidden_states = np.zeros((N_AGENTS, HIDDEN_DIM), dtype=np.float32)

    episode_reward = 0
    episode_deliveries = 0

    if episode < epsilon_decay_steps:
        epsilon = epsilon_start - episode * ((epsilon_start - epsilon_end) / epsilon_decay_steps)
    else:
        epsilon = epsilon_end

    for t in range(MAX_STEPS):
        # --- DECENTRALIZED EXECUTION (BATCHED) ---
        agent_id_oh = EYE_AGENTS
        prev_act_oh = EYE_ACTIONS[np.array(previous_actions, dtype=np.int32)]
        obs_array = np.array(observations, dtype=np.float32)

        combined_input = np.concatenate([agent_id_oh, obs_array, prev_act_oh], axis=1)

        # ---> UPDATED: Inference & Action Selection now passes and returns hidden states <---
        q_values_batch, next_hidden_states = agent_network(
            tf.convert_to_tensor(combined_input, dtype=tf.float32),
            tf.convert_to_tensor(hidden_states, dtype=tf.float32)
        )
        current_actions_tensor = get_actions_tf(q_values_batch, tf.constant(epsilon, dtype=tf.float32))
        current_actions = current_actions_tensor.numpy().tolist()

        # --- ENVIRONMENT STEP ---
        next_observations, rewards, dones, info = env.step(current_actions)
        next_global_state = get_global_state(next_observations)

        episode_reward += sum(rewards)
        episode_deliveries += sum(1 for r in rewards if r > 0.9)

        # --- STORE EXPERIENCE ---
        D_obs[D_ptr] = observations
        D_actions[D_ptr] = current_actions
        D_rewards[D_ptr] = rewards
        D_next_obs[D_ptr] = next_observations
        D_states[D_ptr] = global_state
        D_next_states[D_ptr] = next_global_state
        D_dones[D_ptr] = float(any(dones))
        D_prev_act[D_ptr] = previous_actions

        # ---> NEW: Store the hidden states in the replay buffer <---
        D_hidden[D_ptr] = hidden_states
        D_next_hidden[D_ptr] = next_hidden_states.numpy()

        D_ptr = (D_ptr + 1) % D_MAX
        D_size = min(D_size + 1, D_MAX)

        observations = next_observations
        global_state = next_global_state
        previous_actions = current_actions

        # ---> NEW: Advance the hidden state for the next timestep <---
        hidden_states = next_hidden_states.numpy()

        if all(dones):
            break

    training_rewards.append(episode_reward)
    training_deliveries.append(episode_deliveries)

    # --- CENTRALIZED TRAINING (FIXED FOR QMIX) ---
    # QMIX requires frequent updates to propagate TD errors effectively. Train EVERY episode instead of batching updates like MAPPO.
    if D_size >= MIN_REPLAY_SIZE:
        # ---> FIX: Revert to 4 updates per episode instead of 1<---
        for batch_idx in range(4):
            batch_indices = np.random.randint(0, D_size, size=BATCH_SIZE)

            b_obs = D_obs[batch_indices]
            b_actions = np.expand_dims(D_actions[batch_indices], -1)
            # Sum individual rewards for the global Q-tot target
            b_rewards = D_rewards[batch_indices][:, 0:1]
            b_next_obs = D_next_obs[batch_indices]
            b_states = D_states[batch_indices]
            b_next_states = D_next_states[batch_indices]
            b_dones = np.expand_dims(D_dones[batch_indices], -1)
            b_prev_act = D_prev_act[batch_indices]

            # ---> ADD THESE TWO LINES <---
            b_hidden = D_hidden[batch_indices]
            b_next_hidden = D_next_hidden[batch_indices]

            b_agent_id_oh = np.broadcast_to(np.expand_dims(EYE_AGENTS, 0), (BATCH_SIZE, N_AGENTS, N_AGENTS))
            b_prev_act_oh = EYE_ACTIONS[b_prev_act]

            # Pass raw tensors directly to the GPU graph
            train_step(
                tf.convert_to_tensor(b_obs, dtype=tf.float32),
                tf.convert_to_tensor(b_prev_act_oh, dtype=tf.float32),
                tf.convert_to_tensor(b_actions, dtype=tf.int32),
                tf.convert_to_tensor(b_rewards, dtype=tf.float32),
                tf.convert_to_tensor(b_next_obs, dtype=tf.float32),
                tf.convert_to_tensor(b_states, dtype=tf.float32),
                tf.convert_to_tensor(b_next_states, dtype=tf.float32),
                tf.convert_to_tensor(b_dones, dtype=tf.float32),
                tf.convert_to_tensor(b_agent_id_oh, dtype=tf.float32),
                tf.convert_to_tensor(b_hidden, dtype=tf.float32),
                tf.convert_to_tensor(b_next_hidden, dtype=tf.float32)
            )

    if episode % TARGET_UPDATE_INTERVAL == 0:
        target_agent.set_weights(agent_network.get_weights())
        target_mixer.set_weights(mixer_network.get_weights())

    # ==========================================
    # HEARTBEAT LOG (Prevents Colab from timing out)
    # ==========================================
    if (episode + 1) % 100 == 0 and (episode + 1) % EVAL_INTERVAL != 0:
        print(f"Working... Episode {episode+1}/{NUM_EPISODES} completed.")

    # ==========================================
    # PERIODIC EVALUATION (MAPPO-style)
    # ==========================================
    if (episode + 1) % EVAL_INTERVAL == 0:
        elapsed_time = time.time() - start_time
        print(f"\n--- Evaluation at Episode {episode+1}/{NUM_EPISODES} | Time: {elapsed_time:.1f}s | Epsilon: {epsilon:.2f} ---")

        eval_rewards = []
        eval_successes = []

        for eval_ep in range(EVAL_EPISODES):
            obs = env.reset()
            prev_actions = np.zeros(N_AGENTS)

            # ---> NEW: Initialize hidden state for evaluation episode <---
            hidden_states = np.zeros((N_AGENTS, HIDDEN_DIM), dtype=np.float32)

            ep_reward = 0
            ep_deliveries = 0

            for eval_t in range(MAX_STEPS):
                agent_id_oh = EYE_AGENTS
                prev_act_oh = EYE_ACTIONS[np.array(prev_actions, dtype=np.int32)]
                obs_array = np.array(obs, dtype=np.float32)
                combined_input = np.concatenate([agent_id_oh, obs_array, prev_act_oh], axis=1)

                # ---> UPDATED: Pass and receive hidden states during evaluation <---
                q_values_batch, next_hidden_states = agent_network(
                    tf.convert_to_tensor(combined_input, dtype=tf.float32),
                    tf.convert_to_tensor(hidden_states, dtype=tf.float32)
                )

                # Greedy action selection (no exploration during eval)
                current_actions = tf.argmax(q_values_batch, axis=1).numpy().tolist()

                obs, rewards, dones, info = env.step(current_actions)
                prev_actions = current_actions

                # ---> NEW: Advance hidden state <---
                hidden_states = next_hidden_states.numpy()

                ep_reward += sum(rewards)
                ep_deliveries += sum(1 for r in rewards if r > 0.9)

                if all(dones):
                    break

            eval_rewards.append(ep_reward)
            eval_successes.append(1 if ep_deliveries > 0 else 0)

        eval_success_rate = np.mean(eval_successes)
        eval_mean_reward = np.mean(eval_rewards)

        print(f"Success Rate (eval): {eval_success_rate:.2%} | Mean Reward: {eval_mean_reward:.2f}")
        print(f"Training Mean: {np.mean(training_rewards[-EVAL_INTERVAL:]) if len(training_rewards) >= EVAL_INTERVAL else np.mean(training_rewards):.2f}")

        # Save best weights based on evaluation reward (like MAPPO)
        if eval_mean_reward > best_eval_reward:
            best_eval_reward = eval_mean_reward
            os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
            agent_network.save_weights(os.path.join(CHECKPOINTS_DIR, f"qmix_best_agent.weights.h5"))
            mixer_network.save_weights(os.path.join(CHECKPOINTS_DIR, f"qmix_best_mixer.weights.h5"))
            print(f"✓ New best model saved (reward: {eval_mean_reward:.2f})")

            # Save a snapshot of these peak metrics for the final comparison
            peak_metrics = {
                "peak_episode": episode + 1,
                "peak_eval_reward": float(eval_mean_reward),
                "peak_success_rate": float(eval_success_rate),
                "eval_rewards_list": [float(r) for r in eval_rewards],
                "eval_deliveries_list": [int(d) for d in eval_successes]
            }
            with open(os.path.join(LOGS_DIR, "qmix_peak_metrics.json"), "w") as f:
                json.dump(peak_metrics, f, indent=2)

            print(f"✓ New best model and metrics saved (reward: {eval_mean_reward:.2f})")

        print()

    if (episode + 1) % 1000 == 0:
        os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
        agent_network.save_weights(os.path.join(CHECKPOINTS_DIR, f"qmix_tf_agent_ep{episode+1}.weights.h5"))
        mixer_network.save_weights(os.path.join(CHECKPOINTS_DIR, f"qmix_tf_mixer_ep{episode+1}.weights.h5"))
        print(f"--- Checkpoint saved at Episode {episode+1} ---")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
agent_network.save_weights(os.path.join(CHECKPOINTS_DIR, "qmix_tf_agent.weights.h5"))
mixer_network.save_weights(os.path.join(CHECKPOINTS_DIR, "qmix_tf_mixer.weights.h5"))
print("Training finished! Weights saved.")


# ==========================================
# Save Training Logs (JSON & CSV to match MAPPO)
# ==========================================
# Uses your existing LOGS_DIR (which uses BASE_DIR)
os.makedirs(LOGS_DIR, exist_ok=True)

# 1. Save CSV
training_csv_path = os.path.join(LOGS_DIR, "qmix_train_rewards.csv")
with open(training_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Episode", "Reward", "Deliveries"])
    for ep, (r, d) in enumerate(zip(training_rewards, training_deliveries)):
        writer.writerow([ep, float(r), int(d)])

# 2. Save JSON
training_json_path = os.path.join(LOGS_DIR, "qmix_train_rewards.json")
with open(training_json_path, "w") as f:
    json.dump({
        "_rewards": training_rewards,
        "_deliveries": training_deliveries
    }, f, indent=2)

print(f"Saved QMIX training logs to {LOGS_DIR}")

/usr/local/lib/python3.12/dist-packages/gymnasium/utils/passive_env_checker.py:245: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'list'>
  logger.warn(


Working... Episode 100/6000 completed.
Working... Episode 200/6000 completed.
Working... Episode 300/6000 completed.
Working... Episode 400/6000 completed.
Working... Episode 500/6000 completed.
Working... Episode 600/6000 completed.
Working... Episode 700/6000 completed.
Working... Episode 800/6000 completed.
Working... Episode 900/6000 completed.

--- Evaluation at Episode 1000/6000 | Time: 1616.9s | Epsilon: 0.74 ---
Success Rate (eval): 0.00% | Mean Reward: 0.02
Training Mean: 7.05
✓ New best model saved (reward: 0.02)
✓ New best model and metrics saved (reward: 0.02)

--- Checkpoint saved at Episode 1000 ---
Working... Episode 1100/6000 completed.
Working... Episode 1200/6000 completed.
Working... Episode 1300/6000 completed.
Working... Episode 1400/6000 completed.
Working... Episode 1500/6000 completed.
Working... Episode 1600/6000 completed.
Working... Episode 1700/6000 completed.
Working... Episode 1800/6000 completed.
Working... Episode 1900/6000 completed.

--- Evaluation at 

In [3]:
# ==========================================
# Evaluation vs MAPPO & Random Baseline
# ==========================================
EVAL_EPISODES = 300
print(f"Evaluating QMIX for {EVAL_EPISODES} episodes...")

# ---> ADD THIS LINE TO LOAD YOUR PEAK PERFORMANCE WEIGHTS <---
agent_network.load_weights(os.path.join(CHECKPOINTS_DIR, "qmix_best_agent.weights.h5"))
mixer_network.load_weights(os.path.join(CHECKPOINTS_DIR, "qmix_best_mixer.weights.h5"))

q_r, q_d = [], []

for ep in range(EVAL_EPISODES):
    observations = env.reset()
    previous_actions = np.zeros(N_AGENTS)

    # ---> NEW: Initialize hidden state for final evaluation <---
    hidden_states = np.zeros((N_AGENTS, HIDDEN_DIM), dtype=np.float32)

    episode_reward = 0
    episode_deliveries = 0

    for t in range(MAX_STEPS):
        current_actions = []
        agent_id_oh = EYE_AGENTS
        prev_act_oh = EYE_ACTIONS[np.array(previous_actions, dtype=np.int32)]
        obs_array = np.array(observations, dtype=np.float32)

        combined_input = np.concatenate([agent_id_oh, obs_array, prev_act_oh], axis=1)

        # ---> UPDATED: Network inference with hidden states <---
        # Note:It must split the output tuple before calling .numpy()
        q_values_tensor, next_hidden_states = agent_network(
            tf.convert_to_tensor(combined_input, dtype=tf.float32),
            tf.convert_to_tensor(hidden_states, dtype=tf.float32)
        )
        q_values_batch = q_values_tensor.numpy()

        for i in range(N_AGENTS):
            action_i = int(np.argmax(q_values_batch[i]))
            current_actions.append(action_i)

        observations, rewards, dones, info = env.step(current_actions)
        previous_actions = current_actions

        # ---> NEW: Advance hidden state <---
        hidden_states = next_hidden_states.numpy()

        episode_reward += sum(rewards)
        episode_deliveries += sum(1 for r in rewards if r > 0.9)

        if all(dones):
            break

    q_r.append(episode_reward)
    q_d.append(episode_deliveries)

qmix_mean_reward = np.mean(q_r)
print(f"QMIX Mean Reward: {qmix_mean_reward:.4f}")

def load_baseline(filepath):
    r_list, d_list = [], []
    try:
        with open(filepath) as f:
            data = json.load(f)
        if "episodes" in data:
            r_list = [ep["team_total_reward"] for ep in data["episodes"]][:EVAL_EPISODES]
            # For warehouse env: count deliveries (rewards >= 0.9 indicate successful deliveries)
            # If episodes have explicit deliveries field, use it; otherwise count from rewards
            if "deliveries" in data["episodes"][0]:
                d_list = [ep["deliveries"] for ep in data["episodes"]][:EVAL_EPISODES]
            else:
                # Count deliveries from agent rewards (warehouse gives ~1.0 per delivery)
                d_list = []
                for ep in data["episodes"][:EVAL_EPISODES]:
                    if "agent_total_rewards" in ep:
                        # Count rewards >= 0.9 as successful deliveries
                        deliveries = sum(1 for r in ep["agent_total_rewards"] if r >= 0.9)
                        d_list.append(deliveries)
                    else:
                        # Fallback: assume team_total_reward is delivery count (incorrect but better than nothing)
                        d_list.append(int(ep["team_total_reward"]))
        else:
            r_list = data.get("_rewards", [])[:EVAL_EPISODES]
            d_list = data.get("_deliveries", [])[:EVAL_EPISODES]
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        pass
    return r_list, d_list

m_r, m_d = load_baseline(os.path.join(LOGS_DIR, "trained_policy_rewards.json"))
if not m_r: print("Warning: MAPPO baseline failed to load.")

r_r, r_d = load_baseline(os.path.join(LOGS_DIR, "random_baseline_rewards.json"))
if not r_r: print("Warning: Random baseline failed to load.")

g_r, g_d = load_baseline(os.path.join(LOGS_DIR, "greedy_baseline_rewards.json"))
if not g_r: print("Warning: Greedy baseline failed to load.")

os.makedirs(LOGS_DIR, exist_ok=True)

log_data = {"_rewards": q_r, "_deliveries": q_d, "mean_reward": float(qmix_mean_reward)}
with open(os.path.join(LOGS_DIR, "qmix_evaluation_rewards.json"), "w") as f:
    json.dump(log_data, f, indent=2)

with open(os.path.join(LOGS_DIR, "qmix_evaluation_rewards.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Episode", "Reward", "Deliveries"])
    for ep, (r, d) in enumerate(zip(q_r, q_d)):
        writer.writerow([ep, float(r), int(d)])

print(f"Saved QMIX logs to {os.path.join(LOGS_DIR, 'qmix_evaluation_rewards.json')} and .csv")

summary_csv_path = os.path.join(LOGS_DIR, "qmix_evaluation_summary.csv")
with open(summary_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Policy", "Mean Reward", "Std Reward", "Avg Deliveries", "Max Deliveries", "Delivery Rate (%)", "Positive Reward Rate (%)"])

    q_d_any = sum(1 for d in q_d if d > 0)
    q_pos = sum(1 for r in q_r if r > 0.0)
    writer.writerow([
        "QMIX",
        f"{np.mean(q_r):.4f}",
        f"{np.std(q_r):.4f}",
        f"{np.mean(q_d):.3f}",
        f"{np.max(q_d)}",
        f"{(q_d_any/EVAL_EPISODES)*100:.1f}",
        f"{(q_pos/EVAL_EPISODES)*100:.1f}"
    ])

    if len(m_r) > 1 and len(m_d) > 1:
        m_d_any = sum(1 for d in m_d if d > 0)
        m_pos = sum(1 for r in m_r if r > 0.0)
        writer.writerow(["MAPPO", f"{np.mean(m_r):.4f}", f"{np.std(m_r):.4f}", f"{np.mean(m_d):.3f}", f"{np.max(m_d)}", f"{(m_d_any/len(m_d))*100:.1f}", f"{(m_pos/len(m_r))*100:.1f}"])

    if len(r_r) > 1 and len(r_d) > 1:
        r_d_any = sum(1 for d in r_d if d > 0)
        r_pos = sum(1 for r in r_r if r > 0.0)
        writer.writerow(["Random", f"{np.mean(r_r):.4f}", f"{np.std(r_r):.4f}", f"{np.mean(r_d):.3f}", f"{np.max(r_d)}", f"{(r_d_any/len(r_d))*100:.1f}", f"{(r_pos/len(r_r))*100:.1f}"])

    if len(g_r) > 1 and len(g_d) > 1:
        g_d_any = sum(1 for d in g_d if d > 0)
        g_pos = sum(1 for r in g_r if r > 0.0)
        writer.writerow(["Greedy", f"{np.mean(g_r):.4f}", f"{np.std(g_r):.4f}", f"{np.mean(g_d):.3f}", f"{np.max(g_d)}", f"{(g_d_any/len(g_d))*100:.1f}", f"{(g_pos/len(g_r))*100:.1f}"])

print(f"Saved QMIX Evaluation Summary to {summary_csv_path}")

try:
    from scipy.stats import ttest_ind

    try:
        with open(os.path.join(LOGS_DIR, "qmix_peak_metrics.json"), "r") as f:
            peak_data = json.load(f)
            # Use the rewards list from your 300-episode peak evaluation
            q_r_for_test = peak_data.get("eval_rewards_list", q_r)
            peak_ep = peak_data.get("peak_episode", "Unknown")
    except (FileNotFoundError, KeyError):
        print("Peak metrics not found, falling back to final evaluation data.")
        q_r_for_test = q_r
        peak_ep = "Final"

    with open(os.path.join(LOGS_DIR, "significance_tests.txt"), "w") as sig_f:
        sig_f.write(f"Statistical Significance Tests (QMIX Peak at Ep {peak_ep} vs Baselines)\n")
        sig_f.write("="*70 + "\n")

        # Compare Peak QMIX vs MAPPO
        if len(m_r) > 1:
            t_stat, p_val = ttest_ind(q_r_for_test, m_r, equal_var=False)
            res = f"--- QMIX (PEAK) vs MAPPO ---\nT-statistic: {t_stat:.4f} | P-value: {p_val:.4f} (Signif: {'YES' if p_val < 0.05 else 'NO'})\n"
            print(res); sig_f.write(res + "\n")

        # Compare Peak QMIX vs Random
        if len(r_r) > 1:
            t_stat2, p_val2 = ttest_ind(q_r_for_test, r_r, equal_var=False)
            res2 = f"--- QMIX (PEAK) vs Random ---\nT-statistic: {t_stat2:.4f} | P-value: {p_val2:.4f} (Signif: {'YES' if p_val2 < 0.05 else 'NO'})\n"
            print(res2); sig_f.write(res2 + "\n")

    print(f"Saved Statistical tests to {os.path.join(LOGS_DIR, 'significance_tests.txt')}")
except ImportError:
    print("scipy not found. Skipping statistical tests.")

Evaluating QMIX for 300 episodes...
QMIX Mean Reward: 0.0503
Error loading /content/smart-warehouse/results_20260414_1633/logs/trained_policy_rewards.json: [Errno 2] No such file or directory: '/content/smart-warehouse/results_20260414_1633/logs/trained_policy_rewards.json'
Error loading /content/smart-warehouse/results_20260414_1633/logs/random_baseline_rewards.json: [Errno 2] No such file or directory: '/content/smart-warehouse/results_20260414_1633/logs/random_baseline_rewards.json'
Error loading /content/smart-warehouse/results_20260414_1633/logs/greedy_baseline_rewards.json: [Errno 2] No such file or directory: '/content/smart-warehouse/results_20260414_1633/logs/greedy_baseline_rewards.json'
Saved QMIX logs to /content/smart-warehouse/results_20260414_1633/logs/qmix_evaluation_rewards.json and .csv
Saved QMIX Evaluation Summary to /content/smart-warehouse/results_20260414_1633/logs/qmix_evaluation_summary.csv
Saved Statistical tests to /content/smart-warehouse/results_20260414_16

In [4]:
# ==========================================
# Tri-Model 6-Panel Visualization Dashboard
# ==========================================
q_r = np.array(q_r)
r_r = np.array(r_r)
m_r = np.array(m_r) if len(m_r) > 0 else np.array([])
g_r = np.array(g_r) if len(g_r) > 0 else np.array([])
q_d = np.array(q_d)
r_d = np.array(r_d)
m_d = np.array(m_d) if len(m_d) > 0 else np.array([])
g_d = np.array(g_d) if len(g_d) > 0 else np.array([])

QMIX_COLOR = "#00BCD4"
MAPPO_COLOR = "#AB47BC"
GREEDY_COLOR = "#FF9800"
RAND_COLOR = "#E0E0E0"

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("QMIX (TF) vs MAPPO vs Random Baseline Evaluation", fontsize=16, fontweight="bold")

# Panel 1: Histogram
ax = axes[0, 0]
all_vals = [q_r, r_r]
if len(m_r) > 0: all_vals.append(m_r)
if len(g_r) > 0: all_vals.append(g_r)
bins = np.histogram_bin_edges(np.concatenate(all_vals), bins=20)
ax.hist(q_r, bins=bins, histtype="step", linewidth=2, color=QMIX_COLOR, label=f"QMIX (μ={q_r.mean():.2f})")
if len(m_r) > 0:
    ax.hist(m_r, bins=bins, histtype="step", linewidth=2, color=MAPPO_COLOR, label=f"MAPPO (μ={m_r.mean():.2f})")
if len(g_r) > 0:
    ax.hist(g_r, bins=bins, histtype="step", linewidth=2, color=GREEDY_COLOR, label=f"Greedy (μ={g_r.mean():.2f})")
ax.hist(r_r, bins=bins, histtype="step", linewidth=2, color=RAND_COLOR, label=f"Random (μ={r_r.mean():.2f})")
ax.axvline(q_r.mean(), color=QMIX_COLOR, linewidth=2, linestyle="--")
if len(m_r) > 0: ax.axvline(m_r.mean(), color=MAPPO_COLOR, linewidth=2, linestyle="--")
if len(g_r) > 0: ax.axvline(g_r.mean(), color=GREEDY_COLOR, linewidth=2, linestyle="--")
ax.axvline(r_r.mean(), color="grey", linewidth=2, linestyle="--")
ax.set_title("Reward Distribution")
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: Box Plot
ax = axes[0, 1]
bp_data, bp_labels, bp_colors = [q_r], ["QMIX"], [QMIX_COLOR]
if len(m_r) > 0: bp_data.append(m_r); bp_labels.append("MAPPO"); bp_colors.append(MAPPO_COLOR)
if len(g_r) > 0: bp_data.append(g_r); bp_labels.append("Greedy"); bp_colors.append(GREEDY_COLOR)
bp_data.append(r_r); bp_labels.append("Random"); bp_colors.append(RAND_COLOR)

bp = ax.boxplot(bp_data, tick_labels=bp_labels, patch_artist=True)
for patch, color in zip(bp["boxes"], bp_colors):
    patch.set_facecolor(color)
ax.set_title("Reward Box Plot")
ax.grid(alpha=0.3)

# Panel 3: Mean Deliveries
ax = axes[0, 2]
labels, means, stds, colors = ["QMIX"], [q_d.mean()], [q_d.std()], [QMIX_COLOR]
if len(m_d) > 0: labels.append("MAPPO"); means.append(m_d.mean()); stds.append(m_d.std()); colors.append(MAPPO_COLOR)
if len(g_d) > 0: labels.append("Greedy"); means.append(g_d.mean()); stds.append(g_d.std()); colors.append(GREEDY_COLOR)
labels.append("Random"); means.append(r_d.mean()); stds.append(r_d.std()); colors.append(RAND_COLOR)

bars = ax.bar(labels, means, color=colors, alpha=0.8, yerr=stds, capsize=8)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, m + max(stds)*0.05, f"{m:.2f}", ha="center", va="bottom")
ax.set_title("Mean Deliveries per Episode")
ax.grid(alpha=0.3)

# Panel 4: Training Curve Comparison
ax = axes[1, 0]
window = max(100, NUM_EPISODES // 100)
if len(training_rewards) >= window:
    step_size = max(1, window//2)
    smoothed_qmix = [np.mean(training_rewards[i:i+window]) for i in range(0, len(training_rewards)-window+1, step_size)]
    x_q = np.arange(len(smoothed_qmix)) * step_size
    ax.plot(x_q, smoothed_qmix, color=QMIX_COLOR, linewidth=2, label="QMIX Running Reward")

# ---> NEW: Load and Plot the Peak Performance Star <---
try:
    with open(os.path.join(LOGS_DIR, "qmix_peak_metrics.json"), "r") as f:
        peak_data = json.load(f)
        peak_ep = peak_data["peak_episode"]
        peak_rew = peak_data["peak_eval_reward"]
        ax.scatter(peak_ep, peak_rew, color="gold", s=200, marker="*", edgecolors="black", zorder=5, label=f"Peak (Ep {peak_ep})")
        ax.annotate(f"Peak: {peak_rew:.2f}", (peak_ep, peak_rew), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold')
except FileNotFoundError:
    pass

ax.axhline(r_r.mean(), color=RAND_COLOR, linestyle="--", label="Random Mean")
if len(m_r) > 0: ax.axhline(m_r.mean(), color=MAPPO_COLOR, linestyle="-.", label="MAPPO Mean (Eval)")
if len(g_r) > 0: ax.axhline(g_r.mean(), color=GREEDY_COLOR, linestyle=":", label="Greedy Mean (Eval)")

ax.set_title("Estimated Training Curves overlay")
ax.legend()
ax.grid(alpha=0.3)

# Panel 5: Cumulative Deliveries
ax = axes[1, 1]
ax.plot(np.cumsum(q_d), color=QMIX_COLOR, linewidth=2, label="QMIX (Eval)")
if len(m_d) > 0: ax.plot(np.cumsum(m_d), color=MAPPO_COLOR, linewidth=2, label="MAPPO (Eval)")
if len(g_d) > 0: ax.plot(np.cumsum(g_d), color=GREEDY_COLOR, linewidth=2, label="Greedy (Eval)")
ax.plot(np.cumsum(r_d), color=RAND_COLOR, linewidth=2, label="Random (Eval)")

ax.set_title("Cumulative Deliveries")
ax.legend()
ax.grid(alpha=0.3)

# Panel 6: Positive Rate (Delivery Success)
ax = axes[1, 2]
eval_window = 50
q_pos = [np.mean([1 if d > 0 else 0 for d in q_d[i:i+eval_window]]) for i in range(0, len(q_d)-eval_window+1, eval_window//2)]
r_pos = [np.mean([1 if d > 0 else 0 for d in r_d[i:i+eval_window]]) for i in range(0, len(r_d)-eval_window+1, eval_window//2)]
x_q_pos = np.arange(len(q_pos)) * (eval_window//2)

ax.plot(x_q_pos, q_pos, color=QMIX_COLOR, linewidth=2, label="QMIX")
if len(m_d) > 0:  # Plot MAPPO if we have any data (even if less than eval_window)
    if len(m_d) >= eval_window:
        m_pos = [np.mean([1 if d > 0 else 0 for d in m_d[i:i+eval_window]]) for i in range(0, len(m_d)-eval_window+1, eval_window//2)]
        ax.plot(x_q_pos[:len(m_pos)], m_pos, color=MAPPO_COLOR, linewidth=2, label="MAPPO")
    else:
        # For small datasets, just plot the overall success rate as a horizontal line
        m_success_rate = np.mean([1 if d > 0 else 0 for d in m_d])
        ax.axhline(y=m_success_rate, color=MAPPO_COLOR, linewidth=2, linestyle='--', label=f"MAPPO (μ={m_success_rate:.2%})")
if len(g_d) > eval_window:  # Only plot if we have enough greedy data
    g_pos_d = [np.mean([1 if d > 0 else 0 for d in g_d[i:i+eval_window]]) for i in range(0, len(g_d)-eval_window+1, eval_window//2)]
    ax.plot(x_q_pos[:len(g_pos_d)], g_pos_d, color=GREEDY_COLOR, linewidth=2, label="Greedy")
ax.plot(x_q_pos[:len(r_pos)], r_pos, color=RAND_COLOR, linewidth=2, label="Random")

ax.set_ylim(-0.05, 1.05)
ax.set_title("Delivery Success Rate (Rolling 50-ep)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
os.makedirs(PLOTS_DIR, exist_ok=True)
plt.savefig(os.path.join(PLOTS_DIR, "qmix_comparison_dashboard.png"), dpi=150)
plt.show()


print("\n" + "="*50)
print("Training and evaluation complete!")
print("="*50)
print(f"Results saved to: {RESULTS_DIR}")
print(f"Logs saved to: {LOGS_DIR}")
print(f"Plots saved to: {PLOTS_DIR}")


/tmp/ipykernel_2120/3799067510.py:32: RuntimeWarning: Mean of empty slice.
  ax.hist(r_r, bins=bins, histtype="step", linewidth=2, color=RAND_COLOR, label=f"Random (μ={r_r.mean():.2f})")
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_2120/3799067510.py:36: RuntimeWarning: Mean of empty slice.
  ax.axvline(r_r.mean(), color="grey", linewidth=2, linestyle="--")
/tmp/ipykernel_2120/3799067510.py:59: RuntimeWarning: Mean of empty slice.
  labels.append("Random"); means.append(r_d.mean()); stds.append(r_d.std()); colors.append(RAND_COLOR)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out


Training and evaluation complete!
Results saved to: /content/smart-warehouse/results_20260414_1633
Logs saved to: /content/smart-warehouse/results_20260414_1633/logs
Plots saved to: /content/smart-warehouse/results_20260414_1633/plots


In [5]:
import shutil
from google.colab import files

# Use the absolute path to zip the directory safely
shutil.make_archive('/content/smart-warehouse_results', 'zip', '/content/smart-warehouse')

# Download the file
files.download('/content/smart-warehouse_results.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>